# Fine-tune Gemma for Coding Agent Tasks using LoRA


## Overview

Gemma is a family of lightweight, state-of-the-art open models built from the same research and technology used to create the Gemini models.

Large Language Models (LLMs) like Gemma have been shown to be effective at a variety of NLP tasks, including code generation and reasoning. An LLM is first pre-trained on a large corpus of text and code in a self-supervised fashion. Pre-training helps LLMs learn general-purpose knowledge and coding patterns. An LLM can then be fine-tuned with domain-specific data to perform downstream tasks (such as writing, debugging, and explaining code).

LLMs are extremely large in size (parameters in the order of billions). Full fine-tuning (which updates all the parameters in the model) is not required for most applications because typical fine-tuning datasets are relatively much smaller than the pre-training datasets.

[Low Rank Adaptation (LoRA)](https://arxiv.org/abs/2106.09685){:.external} is a fine-tuning technique which greatly reduces the number of trainable parameters for downstream tasks by freezing the weights of the model and inserting a smaller number of new weights into the model. This makes training with LoRA much faster and more memory-efficient, and produces smaller model weights (a few hundred MBs), all while maintaining the quality of the model outputs.

This tutorial walks you through using KerasNLP to perform LoRA fine-tuning on a Gemma 2B model using the [CodeAlpaca-20k dataset](https://huggingface.co/datasets/sahil2801/CodeAlpaca-20k){:.external}. This dataset contains 20,000 high-quality coding instruction/response pairs — ideal for training a coding agent.


### Install dependencies

Install Keras, KerasNLP, and other dependencies.

In [1]:
# Install Keras 3 last. See https://keras.io/getting_started/ for more details.
!pip install -q -U keras-nlp
!pip install -q -U keras>=3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 21.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 65.0 MB/s eta 0:00:00


In [2]:
import os

os.environ["KERAS_BACKEND"] = "jax"  # Or "torch" or "tensorflow".
# Avoid memory fragmentation on JAX backend.
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"]="1.00"

In [3]:
import keras
import keras_nlp

2026-04-21 10:35:50.070941: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776767750.267655      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776767750.317737      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776767750.745348      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776767750.745391      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776767750.745394      55 computation_placer.cc:177] computation placer alr

## Load Dataset


Preprocess the data. This tutorial uses a subset of 1000 training examples to execute the notebook faster. Consider using more training data for higher quality fine-tuning.

The **CodeAlpaca-20k** dataset is loaded directly from Hugging Face Hub. Each example contains an `instruction` (coding task), an optional `input` (extra context or starter code), and an `output` (the expected code/solution).


In [4]:
# Install the datasets library to load CodeAlpaca from Hugging Face
!pip install -q datasets

from datasets import load_dataset

# Load the CodeAlpaca-20k dataset
ds = load_dataset("sahil2801/CodeAlpaca-20k", split="train")

data = []
template = "Instruction:\n{instruction}\n\nResponse:\n{response}"

for example in ds:
    instruction = example["instruction"]
    # Append input context to instruction if present
    if example.get("input"):
        instruction = instruction + "\n" + example["input"]
    response = example["output"]
    data.append(template.format(instruction=instruction, response=response))

# Only use 1000 training examples, to keep it fast.
# data = data[:1000]
print(f"Loaded {len(data)} examples. Sample:\n")
print(data[0])


README.md:   0%|          | 0.00/147 [00:00<?, ?B/s]

code_alpaca_20k.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/20022 [00:00<?, ? examples/s]

Loaded 20022 examples. Sample:

Instruction:
Create an array of length 5 which contains all even numbers between 1 and 10.

Response:
arr = [2, 4, 6, 8, 10]


## Load Model

KerasNLP provides implementations of many popular [model architectures](https://keras.io/api/keras_nlp/models/){:.external}. In this tutorial, you'll create a model using `GemmaCausalLM`, an end-to-end Gemma model for causal language modeling. A causal language model predicts the next token based on previous tokens.

Create the model using the `from_preset` method:

In [7]:
import gc

del gemma_lm   # delete old model
gc.collect()   # free Python memory

3172

In [8]:
gemma_lm = keras_nlp.models.GemmaCausalLM.from_preset(
    "gemma_2b_en",
    dtype="float16"
)
gemma_lm.summary()

Preprocessor: "gemma_causal_lm_preprocessor_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gemma_tokenizer (GemmaTokenizer)                              │                      Vocab size: 256,000 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "gemma_causal_lm_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gemma_backbone                │ (None, None, 2048)        │   2,506,172,416 │ padding_mask[0][0],        │
│ (GemmaBackbone)               │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 256000)      │     524,288,000 │ gemma_backbone[0][0]       │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 2,506,172,416 (4.67 GB)

 Trainable params: 2,506,172,416 (4.67 GB)

 Non-trainable params: 0 (0.00 B)

## Inference before fine tuning

In this section, you will query the model with various prompts to see how it responds.


### Write a Python Function Prompt

Ask the model to write a simple Python utility function.


In [6]:
prompt = template.format(
    instruction="Write a Python function that takes a list of integers and returns the two numbers that add up to a given target sum.",
    response="",
)
print(gemma_lm.generate(prompt, max_length=512))


Instruction:
Write a Python function that takes a list of integers and returns the two numbers that add up to a given target sum.

Response:
def two_sum(nums, target):
    for i in range(len(nums)):
        for j in range(i+1, len(nums)):
            if nums[i] + nums[j] == target:
                return [i, j]
    return None



### Debug Code Prompt

Prompt the model to find and fix a bug in a Python snippet.


In [7]:
prompt = template.format(
    instruction="Debug the following Python code so it correctly reverses a linked list:\n"
    "def reverse_list(head):\n"
    "    prev = None\n"
    "    curr = head\n"
    "    while curr:\n"
    "        curr.next = prev  # bug here\n"
    "        prev = curr\n"
    "        curr = curr.next\n"
    "    return prev",
    response="",
)
print(gemma_lm.generate(prompt, max_length=512))


Instruction:
Debug the following Python code so it correctly reverses a linked list:
def reverse_list(head):
    prev = None
    curr = head
    while curr:
        curr.next = prev  # bug here
        prev = curr
        curr = curr.next
    return prev

Response:
def reverse_list(head):
    prev = None
    curr = head
    while curr:
        curr.next = prev
        prev = curr
        curr = curr.next
    return prev



## LoRA Fine-tuning

To get better code generation responses from the model, fine-tune it with Low Rank Adaptation (LoRA) using the CodeAlpaca-20k dataset.

The LoRA rank determines the dimensionality of the trainable matrices that are added to the original weights of the LLM. It controls the expressiveness and precision of the fine-tuning adjustments.

A higher rank means more detailed changes are possible, but also means more trainable parameters. A lower rank means less computational overhead, but potentially less precise adaptation.

This tutorial uses a LoRA rank of 4. In practice, begin with a relatively small rank (such as 4, 8, 16). This is computationally efficient for experimentation. Train your model with this rank and evaluate the performance improvement on your task. Gradually increase the rank in subsequent trials and see if that further boosts performance.


In [9]:
# 1. Enable LoRA
gemma_lm.backbone.enable_lora(rank=4)

# 2. Set precision explicitly
keras.mixed_precision.set_global_policy("float16")

# 3. Set sequence length
gemma_lm.preprocessor.sequence_length = 256

# 4. Compile
optimizer = keras.optimizers.AdamW(learning_rate=1e-5, weight_decay=0.01)
optimizer.exclude_from_weight_decay(var_names=["bias", "scale"])
gemma_lm.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=optimizer,
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy()],
)

gemma_lm.fit(data, epochs=3, batch_size=1,shuffle=True)

Epoch 1/3
20022/20022 ━━━━━━━━━━━━━━━━━━━━ 3751s 187ms/step - loss: 0.4534 - sparse_categorical_accuracy: 0.7496
Epoch 2/3
20022/20022 ━━━━━━━━━━━━━━━━━━━━ 3739s 187ms/step - loss: 0.4151 - sparse_categorical_accuracy: 0.7636
Epoch 3/3
20022/20022 ━━━━━━━━━━━━━━━━━━━━ 3737s 187ms/step - loss: 0.3885 - sparse_categorical_accuracy: 0.7732


Note that enabling LoRA reduces the number of trainable parameters significantly (from 2.5 billion to 1.3 million).

## Inference after fine-tuning
After fine-tuning, responses follow the coding instruction provided in the prompt.


### Write a Python Function Prompt



In [ ]:
prompt = template.format(
    instruction="Write a Python function that takes a list of integers and returns the two numbers that add up to a given target sum.",
    response="",
)
print(gemma_lm.generate(prompt, max_length=512))


Instruction:
Write a Python function that takes a list of integers and returns the two numbers that add up to a given target sum.

Response:
def two_sum(nums, target):
    for i in range(len(nums)):
        for j in range(i+1, len(nums)):
            if nums[i] + nums[j] == target:
                return [i, j]
    return None



### Debug Code Prompt



In [13]:
prompt = template.format(
    instruction="Debug the following Python code so it correctly reverses a linked list:\n"
    "def reverse_list(head):\n"
    "    prev = None\n"
    "    curr = head\n"
    "    while curr:\n"
    "        curr.next = prev  # bug here\n"
    "        prev = curr\n"
    "        curr = curr.next\n"
    "    return prev",
    response="",
)
print(gemma_lm.generate(prompt, max_length=512))


Instruction:
Debug the following Python code so it correctly reverses a linked list:
def reverse_list(head):
    prev = None
    curr = head
    while curr:
        curr.next = prev  # bug here
        prev = curr
        curr = curr.next
    return prev

Response:
def reverse_list(head):
    prev = None
    curr = head
    while curr:
        curr.next, prev = prev, curr
        curr = curr.next
    return prev



In [18]:
gemma_lm.save_weights("gemma_lora.weights.h5")


In [16]:
import shutil

shutil.make_archive("gemma_lora", 'zip', root_dir=".", base_dir="gemma_lora.weights.h5")

'/kaggle/working/gemma_lora.zip'

In [17]:
from IPython.display import FileLink

FileLink("gemma_lora.zip")

/kaggle/working/gemma_lora.zip

Note that for demonstration purposes, this tutorial fine-tunes the model on a small subset of the dataset for just one epoch and with a low LoRA rank value. To get better code generation responses from the fine-tuned model, you can experiment with:

1. Increasing the size of the fine-tuning dataset (use all 20k CodeAlpaca examples)
2. Training for more steps (epochs)
3. Setting a higher LoRA rank (e.g., 8, 16, or 32)
4. Modifying the hyperparameter values such as `learning_rate` and `weight_decay`.
5. Including examples with `input` context (e.g., starter code) for richer coding scenarios.


## Summary and next steps

This tutorial covered LoRA fine-tuning on a Gemma model for coding agent tasks using the CodeAlpaca-20k dataset and KerasNLP. Check out the following docs next:

* Learn how to [generate text with a Gemma model](https://ai.google.dev/gemma/docs/get_started).
* Learn how to perform [distributed fine-tuning and inference on a Gemma model](https://ai.google.dev/gemma/docs/distributed_tuning).
* Learn how to [use Gemma open models with Vertex AI](https://cloud.google.com/vertex-ai/docs/generative-ai/open-models/use-gemma).
* Explore more coding datasets like [CodeSearchNet](https://huggingface.co/datasets/code_search_net) or [The Stack](https://huggingface.co/datasets/bigcode/the-stack) for larger-scale fine-tuning.
